# Notebook Eksplorasi Dataset & Pra-Pemrosesan (Preprocessing) Sinyal ECG

**Topik Skripsi**: Estimasi Tekanan Darah Non-Invasif Berbasis Sinyal Elektrokardiogram (ECG) Menggunakan Deep Learning  
**Dataset**: *Cuff-Less Blood Pressure Estimation Dataset* (Kachuee et al., 2016 / PhysioNet MIMIC-II) via Kaggle  
**Laju Sampling ($F_s$)**: $125\text{ Hz}$  
**Ukuran Epoch**: $10\text{ detik}$ ($1250\text{ sampel}$ per epoch, *non-overlapping*)

---

### Daftar Isi & Struktur Notebook:
1. **Blok 0 — Setup & Load Semua Part Dataset & Ringkasan Statistik**
   - Pemuatan seluruh file `.mat` (`part_1.mat` s/d `part_12.mat`)
   - Ekstraksi informasi durasi, jumlah sampel, dan beat-by-beat Ground Truth BP
2. **Blok 1 — Visualisasi Distribusi Ground Truth BP (SBP, DBP, MAP)**
   - Histogram & Kernel Density Estimation (KDE) nilai SBP, DBP, dan MAP
   - Boxplot distribusi tekanan darah per part dataset
3. **Blok 2 — Visualisasi Sampel Sinyal Multi-Channel per Part**
   - Plot 3 kanal sinyal simultan: PPG (Row 0), ABP Invasif (Row 1), ECG Lead II (Row 2)
4. **Blok 3 — Epoching & Preprocessing per Tahap (Step-by-Step)**
   - **Blok 3a**: *IIR Notch Filter 50 Hz*
   - **Blok 3b**: *Butterworth Bandpass Filter 0.5–40 Hz*
   - **Blok 3c**: *Min-Max Normalization [0, 1]* (Fan et al., 2021 & Li et al., 2020)
5. **Blok 4 — Eksekusi Batch Preprocessing Seluruh Dataset & Simpan .npz**
   - Pemrosesan *end-to-end* seluruh file `part_*.mat`
   - Ekspor berkas `part_X_preprocessed.npz` siap latih

---

> **Catatan Kaggle**: Notebook ini bersifat *self-contained* — semua fungsi preprocessing didefinisikan inline tanpa dependensi `src/`. Pastikan sudah **Add Data** → `mkachuee/BloodPressureDataset`.

---
## 1. Blok 0 — Setup, Fungsi Preprocessing & Load Dataset

> **Struktur Matriks Dataset Kachuee et al. (per rekaman):**
> - **Baris 0**: Sinyal *Photoplethysmogram* (PPG) @ $125\text{ Hz}$
> - **Baris 1**: Sinyal *Arterial Blood Pressure* (ABP invasif dalam satuan $\text{mmHg}$) @ $125\text{ Hz}$
> - **Baris 2**: Sinyal *Electrocardiogram* (ECG Kanal II) @ $125\text{ Hz}$

In [ ]:
import os
import glob
import gc
import numpy as np
import pandas as pd
import scipy.io
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import iirnotch, butter, filtfilt, find_peaks
from tqdm.auto import tqdm

# ══════════════════════════════════════════════════════════════════════════════
#                          PARAMETER GLOBAL
# ══════════════════════════════════════════════════════════════════════════════
FS = 125.0                      # Sampling rate (Hz)
EPOCH_SEC = 10.0                # Durasi per epoch (detik)
EPOCH_LEN = int(FS * EPOCH_SEC) # 1250 sampel per epoch

# Filter Parameters
NOTCH_FREQ = 50.0   # Frekuensi notch PLN (Hz)
NOTCH_Q = 30.0      # Quality factor
BP_LOWCUT = 0.5     # Bandpass low cutoff (Hz)
BP_HIGHCUT = 40.0   # Bandpass high cutoff (Hz)
BP_ORDER = 2        # Orde Butterworth (efektif orde 4 via filtfilt)

# Filter kualitas BP (rentang fisiologis valid)
SBP_RANGE = (50.0, 220.0)  # mmHg
DBP_RANGE = (30.0, 140.0)  # mmHg

print(f"Konfigurasi Global:")
print(f" - Frekuensi Sampling (FS)  : {FS} Hz")
print(f" - Durasi Epoch             : {EPOCH_SEC} detik ({EPOCH_LEN} sampel)")
print(f" - Notch Filter             : {NOTCH_FREQ} Hz (Q={NOTCH_Q})")
print(f" - Bandpass                 : {BP_LOWCUT}–{BP_HIGHCUT} Hz (Orde {BP_ORDER})")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#                FUNGSI PREPROCESSING (SELF-CONTAINED / INLINE)
# ══════════════════════════════════════════════════════════════════════════════

def iir_notch_filter(signal, fs=125.0, f0=50.0, q=30.0):
    """
    IIR Notch Filter — Eliminasi interferensi frekuensi listrik PLN 50 Hz.
    Metode: Zero-Phase Filtering (filtfilt) sesuai Alhamid (2026) & Hakim (2025).
    """
    if signal.size == 0:
        return signal
    b, a = iirnotch(w0=f0, Q=q, fs=fs)
    return filtfilt(b, a, signal, axis=0)


def butterworth_bandpass_filter(signal, fs=125.0, lowcut=0.5, highcut=40.0, order=2):
    """
    Butterworth Bandpass Filter 0.5–40 Hz.
    Zero-Phase Filtering (filtfilt) → orde efektif = 2 * order = 4.
    """
    if signal.size == 0:
        return signal
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    if high >= 1.0:
        high = 0.99
    b, a = butter(N=order, Wn=[low, high], btype='bandpass')
    return filtfilt(b, a, signal, axis=0)


def minmax_scale(signal, eps=1e-8):
    """
    Min-Max Normalization [0, 1] sesuai Fan et al. (2021) & Li et al. (2020).
    Formula: x_norm = (signal - x_min) / (x_max - x_min + eps)
    """
    if signal.size == 0:
        return signal
    x_min = np.min(signal)
    x_max = np.max(signal)
    range_val = x_max - x_min
    if range_val < eps:
        range_val = 1.0
    return (signal - x_min) / range_val


def preprocess_ecg_pipeline(signal, fs=125.0):
    """Pipeline preprocessing lengkap: Notch → Bandpass → MinMax [0,1]."""
    signal = iir_notch_filter(signal, fs=fs, f0=NOTCH_FREQ, q=NOTCH_Q)
    signal = butterworth_bandpass_filter(signal, fs=fs, lowcut=BP_LOWCUT, highcut=BP_HIGHCUT, order=BP_ORDER)
    signal = minmax_scale(signal)
    return signal


def compute_bp_metrics(abp_signal, fs=125.0):
    """
    Menghitung SBP, DBP, MAP dari sinyal ABP menggunakan deteksi puncak per beat.
    Metode beat-by-beat sesuai standar klinis (bukan global max/min).
    
    Returns: dict dengan keys 'sbp', 'dbp', 'map', 'n_beats', 'valid'
    """
    min_peak_distance = int(0.3 * fs)  # ~37 sampel (HR ≤ 200 bpm)
    min_peak_height = np.percentile(abp_signal, 50)

    systolic_peaks, _ = find_peaks(abp_signal, distance=min_peak_distance, height=min_peak_height)
    diastolic_troughs, _ = find_peaks(-abp_signal, distance=min_peak_distance)

    sbp = float(np.mean(abp_signal[systolic_peaks])) if len(systolic_peaks) > 0 else float(np.max(abp_signal))
    dbp = float(np.mean(abp_signal[diastolic_troughs])) if len(diastolic_troughs) > 0 else float(np.min(abp_signal))
    map_val = float(np.mean(abp_signal))

    is_valid = (
        len(systolic_peaks) > 0
        and SBP_RANGE[0] <= sbp <= SBP_RANGE[1]
        and DBP_RANGE[0] <= dbp <= DBP_RANGE[1]
    )
    return {
        'sbp': sbp, 'dbp': dbp, 'map': map_val,
        'n_beats': len(systolic_peaks), 'valid': is_valid,
        'systolic_peaks_idx': systolic_peaks,
        'diastolic_troughs_idx': diastolic_troughs,
    }

print('✅ Fungsi preprocessing (inline) siap.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#                    FUNGSI PEMUAT DATASET (SELF-CONTAINED)
# ══════════════════════════════════════════════════════════════════════════════

def load_mat_file(file_path):
    """Membaca file .mat (MATLAB v5 / v7 atau HDF5 v7.3)."""
    try:
        mat = scipy.io.loadmat(file_path)
        keys = [k for k in mat.keys() if not k.startswith('__')]
        return mat[keys[0]]
    except Exception:
        import h5py
        with h5py.File(file_path, 'r') as f:
            return np.array(f[list(f.keys())[0]])


def extract_record_matrix(cell_data, index):
    """
    Ekstrak matriks sinyal [3, N_samples] dari cell array.
    Struktur: Row 0=PPG, Row 1=ABP, Row 2=ECG.
    """
    if isinstance(cell_data, np.ndarray) and cell_data.dtype == object:
        raw = cell_data[0, index]
    else:
        raw = cell_data[index]
    matrix = np.array(raw, dtype=np.float64)
    if matrix.ndim == 2 and matrix.shape[0] > matrix.shape[1] and matrix.shape[1] == 3:
        matrix = matrix.T
    return matrix


def get_num_records(cell_data):
    """Hitung jumlah rekaman dalam cell array."""
    if isinstance(cell_data, np.ndarray) and cell_data.dtype == object:
        return cell_data.shape[1]
    return cell_data.shape[0]


print('✅ Fungsi pemuat dataset siap.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#                         PEMUATAN DATASET
# ══════════════════════════════════════════════════════════════════════════════

# Path dataset di Kaggle Input (sudah ter-attach, tanpa download)
PRIMARY_DATA_DIR = '/kaggle/input/datasets/mkachuee/BloodPressureDataset'

if os.path.exists(PRIMARY_DATA_DIR):
    KAGGLE_DATA_DIR = PRIMARY_DATA_DIR
else:
    alt_paths = [
        '/kaggle/input/bloodpressuredataset',
        '/kaggle/input/BloodPressureDataset',
        '/kaggle/input/bloodpressuredataset/BloodPressureDataset',
    ]
    for p in alt_paths:
        if os.path.exists(p):
            KAGGLE_DATA_DIR = p
            break
    else:
        mat_search = sorted(glob.glob('/kaggle/input/**/*.mat', recursive=True))
        if mat_search:
            KAGGLE_DATA_DIR = os.path.dirname(mat_search[0])
        else:
            raise FileNotFoundError(
                f'Dataset tidak ditemukan di {PRIMARY_DATA_DIR} maupun /kaggle/input.\n'
                'Pastikan dataset sudah ditambahkan ke notebook Kaggle via panel Input.'
            )

mat_files = sorted(glob.glob(os.path.join(KAGGLE_DATA_DIR, '**', '*.mat'), recursive=True))
print(f'Direktori dataset: {KAGGLE_DATA_DIR}')
print(f'Ditemukan {len(mat_files)} file .mat:')
for f in mat_files:
    size_mb = os.path.getsize(f) / 1024**2
    print(f'  • {os.path.basename(f)} ({size_mb:.1f} MB)')


In [ ]:
# Ekstraksi Ringkasan Statistik untuk Seluruh File Part Dataset
records_summary = []
loaded_parts = {}

print("Memproses dan mengekstraksi metrik statistik dari setiap part...")
for mat_path in mat_files:
    part_name = os.path.basename(mat_path)
    cell_array = load_mat_file(mat_path)
    if cell_array is None:
        continue
    
    num_records = get_num_records(cell_array)
    
    # Simpan referensi data part untuk visualisasi berikutnya
    loaded_parts[part_name] = cell_array
    
    # Sampling beberapa rekaman untuk menghitung rata-rata metrik Ground Truth BP
    sbp_list, dbp_list, map_list = [], [], []
    sample_step = max(1, num_records // 50)  # sampel representatif ~50 rekaman per part
    
    for idx in range(0, num_records, sample_step):
        try:
            rec = extract_record_matrix(cell_array, idx)
        except Exception:
            continue
        if rec is not None and rec.ndim == 2 and rec.shape[0] >= 3:
            abp_sig = rec[1, :]
            bp = compute_bp_metrics(abp_sig, fs=FS)
            if bp['valid']:
                sbp_list.append(bp['sbp'])
                dbp_list.append(bp['dbp'])
                map_list.append(bp['map'])
    
    records_summary.append({
        'Part': part_name,
        'Jumlah Rekaman': num_records,
        'Mean SBP (mmHg)': np.mean(sbp_list) if sbp_list else np.nan,
        'Std SBP (mmHg)': np.std(sbp_list) if sbp_list else np.nan,
        'Mean DBP (mmHg)': np.mean(dbp_list) if dbp_list else np.nan,
        'Std DBP (mmHg)': np.std(dbp_list) if dbp_list else np.nan,
        'Mean MAP (mmHg)': np.mean(map_list) if map_list else np.nan,
        'Sampel BP Valid': len(sbp_list)
    })

df_summary = pd.DataFrame(records_summary)
print("\n--- Ringkasan Dataset per File Part ---")
display(df_summary)

---
## 2. Blok 1 — Visualisasi Distribusi Ground Truth BP (SBP, DBP, MAP)

Ground Truth target estimasi diperoleh secara invasif melalui kateter arteri (*Arterial Blood Pressure* / ABP).

- **Systolic Blood Pressure (SBP)**: Tekanan puncak saat ventrikel berkontraksi (*systole*).
- **Diastolic Blood Pressure (DBP)**: Tekanan terendah saat relaksasi jantung (*diastole*).
- **Mean Arterial Pressure (MAP)**: $\text{MAP} = \text{DBP} + \frac{1}{3}(\text{SBP} - \text{DBP})$

Nilai referensi klinis menurut pedoman AHA/ACC:
- **Normal**: SBP $< 120\text{ mmHg}$ dan DBP $< 80\text{ mmHg}$
- **Pre-Hipertensi / Elevated**: SBP $120–129\text{ mmHg}$ dan DBP $< 80\text{ mmHg}$
- **Hipertensi Stage 1**: SBP $130–139\text{ mmHg}$ atau DBP $80–89\text{ mmHg}$
- **Hipertensi Stage 2**: SBP $\ge 140\text{ mmHg}$ atau DBP $\ge 90\text{ mmHg}$

In [ ]:
# Ekstraksi agregat Ground Truth BP dari seluruh part yang dimuat
all_sbp, all_dbp, all_map, part_labels = [], [], [], []

for part_name, cell_array in loaded_parts.items():
    num_records = get_num_records(cell_array)
    step = max(1, num_records // 60)
    for idx in range(0, num_records, step):
        try:
            rec = extract_record_matrix(cell_array, idx)
        except Exception:
            continue
        if rec is not None and rec.ndim == 2 and rec.shape[0] >= 3:
            bp = compute_bp_metrics(rec[1, :], fs=FS)
            if bp['valid']:
                all_sbp.append(bp['sbp'])
                all_dbp.append(bp['dbp'])
                all_map.append(bp['map'])
                part_labels.append(part_name)

df_all_bp = pd.DataFrame({
    'Part': part_labels,
    'SBP': all_sbp,
    'DBP': all_dbp,
    'MAP': all_map
})

print(f"Total sampel Ground Truth BP tervalidasi: {len(df_all_bp)}")
print(df_all_bp[['SBP', 'DBP', 'MAP']].describe().round(2))

In [ ]:
# Visualisasi 1: Distribusi Histogram & Densitas SBP, DBP, MAP
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. SBP
axes[0].hist(df_all_bp['SBP'], bins=35, density=True, color='#1f77b4', alpha=0.65, edgecolor='black')
df_all_bp['SBP'].plot(kind='kde', ax=axes[0], color='#0d4b75', lw=2)
axes[0].axvline(df_all_bp['SBP'].mean(), color='red', linestyle='--', lw=1.5, label=f"Mean: {df_all_bp['SBP'].mean():.1f}")
axes[0].axvline(120, color='green', linestyle=':', lw=1.5, label='Batas Normal (120)')
axes[0].axvline(140, color='darkorange', linestyle=':', lw=1.5, label='Batas Hipertensi (140)')
axes[0].set_title("Distribusi Systolic Blood Pressure (SBP)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("SBP (mmHg)"); axes[0].set_ylabel("Densitas")
axes[0].legend(loc='upper right'); axes[0].grid(True, linestyle='--', alpha=0.5)

# 2. DBP
axes[1].hist(df_all_bp['DBP'], bins=35, density=True, color='#2ca02c', alpha=0.65, edgecolor='black')
df_all_bp['DBP'].plot(kind='kde', ax=axes[1], color='#1b611b', lw=2)
axes[1].axvline(df_all_bp['DBP'].mean(), color='red', linestyle='--', lw=1.5, label=f"Mean: {df_all_bp['DBP'].mean():.1f}")
axes[1].axvline(80, color='green', linestyle=':', lw=1.5, label='Batas Normal (80)')
axes[1].axvline(90, color='darkorange', linestyle=':', lw=1.5, label='Batas Hipertensi (90)')
axes[1].set_title("Distribusi Diastolic Blood Pressure (DBP)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("DBP (mmHg)"); axes[1].set_ylabel("Densitas")
axes[1].legend(loc='upper right'); axes[1].grid(True, linestyle='--', alpha=0.5)

# 3. MAP
axes[2].hist(df_all_bp['MAP'], bins=35, density=True, color='#ff7f0e', alpha=0.65, edgecolor='black')
df_all_bp['MAP'].plot(kind='kde', ax=axes[2], color='#b35500', lw=2)
axes[2].axvline(df_all_bp['MAP'].mean(), color='red', linestyle='--', lw=1.5, label=f"Mean: {df_all_bp['MAP'].mean():.1f}")
axes[2].axvline(70, color='green', linestyle=':', lw=1.5, label='Perfusi Min (70)')
axes[2].axvline(105, color='darkorange', linestyle=':', lw=1.5, label='Perfusi Tinggi (105)')
axes[2].set_title("Distribusi Mean Arterial Pressure (MAP)", fontsize=12, fontweight='bold')
axes[2].set_xlabel("MAP (mmHg)"); axes[2].set_ylabel("Densitas")
axes[2].legend(loc='upper right'); axes[2].grid(True, linestyle='--', alpha=0.5)

plt.suptitle("Blok 1: Distribusi Ground Truth Blood Pressure (PhysioNet MIMIC-II)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi 2: Boxplot Homogenitas Tekanan Darah Antar Part Dataset
plt.figure(figsize=(15, 6))
df_melted = df_all_bp.melt(id_vars=['Part'], value_vars=['SBP', 'DBP', 'MAP'], var_name='Target BP', value_name='mmHg')

sns.boxplot(data=df_melted, x='Part', y='mmHg', hue='Target BP', palette=['#1f77b4', '#2ca02c', '#ff7f0e'])
plt.title("Blok 1: Distribusi SBP, DBP, dan MAP per Bagian File Dataset (.mat)", fontsize=13, fontweight='bold')
plt.xlabel("File Dataset"); plt.ylabel("Tekanan Darah (mmHg)")
plt.xticks(rotation=30, ha='right')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(title='Metrik BP', loc='upper right')
plt.tight_layout()
plt.show()

---
## 3. Blok 2 — Visualisasi Sampel Sinyal Multi-Channel per Part

Memeriksa bentuk gelombang (*waveform*) multi-kanal simultan dari setiap file `part_*.mat` untuk memastikan kualitas dan sinkronisasi sinyal:
- **Kanal 0 (PPG)**: Sinyal pulsasi volume darah kapiler perifer.
- **Kanal 1 (ABP Invasif)**: Gelombang tekanan darah arteri kontinu (mmHg).
- **Kanal 2 (ECG Lead II)**: Aktivitas kelistrikan depolarisasi dan repolarisasi jantung.

In [ ]:
# Visualisasi Sinyal Multi-Channel 10 Detik dari Tiap Part Dataset
num_parts = len(loaded_parts)
fig, axes = plt.subplots(num_parts, 1, figsize=(16, 3.2 * num_parts), sharex=False)
if num_parts == 1:
    axes = [axes]

for i, (part_name, cell_array) in enumerate(loaded_parts.items()):
    # Ambil rekaman pertama yang valid
    rec = None
    num_records = get_num_records(cell_array)
    for r_idx in range(min(num_records, 20)):  # cek max 20 rekaman awal
        try:
            candidate = extract_record_matrix(cell_array, r_idx)
        except Exception:
            continue
        if candidate is not None and candidate.ndim == 2 and candidate.shape[0] >= 3 and candidate.shape[1] >= EPOCH_LEN:
            rec = candidate[:, :EPOCH_LEN]  # Ambil jendela 10 detik (1250 sampel)
            break
    
    if rec is None:
        continue
        
    time_axis = np.arange(EPOCH_LEN) / FS
    ppg_sig = rec[0, :]
    abp_sig = rec[1, :]
    ecg_sig = rec[2, :]
    
    ax = axes[i]
    ax_abp = ax.twinx()  # Sumbu kedua untuk ABP mmHg
    
    # Plot ECG dan PPG di sumbu kiri
    l1 = ax.plot(time_axis, ecg_sig, color='#1f77b4', lw=1.2, label='ECG Lead II (Row 2)')
    l2 = ax.plot(time_axis, ppg_sig * (np.std(ecg_sig)/ (np.std(ppg_sig) + 1e-6)) + np.mean(ecg_sig),
                 color='#2ca02c', alpha=0.5, lw=1.0, linestyle='--', label='PPG Sinyal (Row 0, scaled)')
    
    # Plot ABP di sumbu kanan
    l3 = ax_abp.plot(time_axis, abp_sig, color='#d62728', lw=1.3, alpha=0.85, label='ABP Ground Truth (Row 1, mmHg)')
    
    ax.set_ylabel("Amplitudo ECG/PPG (a.u.)", color='#1f77b4', fontweight='bold')
    ax_abp.set_ylabel("ABP (mmHg)", color='#d62728', fontweight='bold')
    ax.set_title(f"Sampel Sinyal Multi-Kanal (10 Detik): {part_name}", fontsize=11, fontweight='bold')
    ax.set_xlabel("Waktu (detik)")
    ax.grid(True, linestyle='--', alpha=0.4)
    
    # Gabungkan legend
    lines = l1 + l2 + l3
    labels = [l.get_label() for l in lines]
    ax.legend(lines, labels, loc='upper right', fontsize=9, framealpha=0.9)

plt.suptitle("Blok 2: Visualisasi Sampel Sinyal Multi-Kanal Simultan per File Part Dataset", fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

---
## 4. Blok 3 — Epoching & Pra-Pemrosesan (Preprocessing) per Tahap

Pra-pemrosesan sinyal ECG mentah dilakukan secara berurutan (*step-by-step*) untuk memastikan kualitas sinyal yang optimal sebelum ekstraksi fitur dan pemodelan deep learning.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        ALUR PIPELINE PREPROCESSING ECG                     │
└─────────────────────────────────────────────────────────────────────────────┘
        │
        ▼
 ┌───────────────┐
 │ Sinyal Mentah │ (ECG Lead II @ 125 Hz, Terkontaminasi Noise)
 └───────┬───────┘
        │
        ▼
 ┌───────────────┐
 │   Epoching    │ (Segmentasi Jendela 10 Detik = 1250 Sampel, Non-Overlapping)
 └───────┬───────┘
        │
        ▼
 ┌───────────────┐
 │    Blok 3a    │  IIR Notch Filter (50 Hz, Q=30)
 │  Notch Filter │  Mengeliminasi noise frekuensi jala-jala listrik PLN 50 Hz
 └───────┬───────┘
        │
        ▼
 ┌───────────────┐
 │    Blok 3b    │  Butterworth Bandpass (0.5 - 40 Hz, Ordo 2)
 │   Bandpass    │  Menghapus baseline wander (<0.5 Hz) & noise EMG (>40 Hz)
 └───────┬───────┘
        │
        ▼
 ┌───────────────┐
 │    Blok 3c    │  Min-Max Normalization [0, 1] (Fan et al., 2021)
 │  Normalisasi  │  Menstandarkan rentang amplitudo ke interval tertutup [0, 1]
 └───────┬───────┘
        │
        ▼
 ┌───────────────┐
 │ Sinyal Bersih │ (Siap Diinputkan ke Model Deep Learning)
 └───────────────┘
```

### 4.0 Setup Epoch Contoh
Kita memilih satu segmen rekaman sepanjang **10 detik ($1250$ sampel)** dari dataset untuk mendemonstrasikan perubahan sinyal pada setiap tahapan pra-pemrosesan.

In [ ]:
# Ekstraksi 1 Epoch Contoh (10 Detik = 1250 Sampel)
sample_part_key = list(loaded_parts.keys())[0]
sample_cell = loaded_parts[sample_part_key]

# Ambil rekaman pertama
rec_sample = extract_record_matrix(sample_cell, 0)
epoch_ecg_raw = rec_sample[2, :EPOCH_LEN]   # Sinyal ECG mentah
epoch_abp_gt  = rec_sample[1, :EPOCH_LEN]   # Ground Truth ABP
epoch_time    = np.arange(EPOCH_LEN) / FS    # Sumbu waktu (detik)

# Visualisasi Epoch Mentah Awal
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
ax1.plot(epoch_time, epoch_ecg_raw, color='#7f7f7f', lw=1.2, label='ECG Mentah (Raw 125 Hz)')
ax1.set_title("Kondisi Awal: Sinyal ECG Mentah (Lead II) Sebelum Preprocessing", fontsize=11, fontweight='bold')
ax1.set_ylabel("Amplitudo Mentah (a.u.)"); ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper right')

ax2.plot(epoch_time, epoch_abp_gt, color='#d62728', lw=1.2, label='Arterial Blood Pressure (ABP Invasif)')
ax2.set_title("Ground Truth ABP Target Estimasi", fontsize=11, fontweight='bold')
ax2.set_xlabel("Waktu (detik)"); ax2.set_ylabel("Tekanan Darah (mmHg)"); ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

---
### 4.1 Blok 3a — IIR Notch Filter 50 Hz

**Tujuan**: Mengeliminasi dengung interferensi jala-jala listrik PLN (*powerline interference*) pada frekuensi $50\text{ Hz}$.

| Parameter | Nilai | Keterangan |
| :--- | :--- | :--- |
| **Frekuensi Notch ($f_0$)** | $50.0\text{ Hz}$ | Frekuensi pusat interferensi listrik AC di Indonesia / standar internasional |
| **Quality Factor ($Q$)** | $30.0$ | Lebar pita penolakan sempit ($BW = f_0 / Q \approx 1.67\text{ Hz}$) untuk menjaga bentuk morfologi ECG |
| **Metode Filtering** | `scipy.signal.filtfilt` | *Zero-phase forward-backward filtering* (tanpa distorsi fasa) |

In [ ]:
# ── Blok 3a: IIR Notch Filter 50 Hz ──────────────────────────────────────────

epoch_step1_notch = iir_notch_filter(epoch_ecg_raw, fs=FS, f0=50.0, q=30.0)

# Visualisasi Komparasi: Mentah vs Notch Filter
fig, ax = plt.subplots(1, 1, figsize=(14, 4))
ax.plot(epoch_time, epoch_ecg_raw, color='gray', alpha=0.6, lw=1.0, label='ECG Mentah (Raw 125 Hz)')
ax.plot(epoch_time, epoch_step1_notch, color='#2ca02c', lw=1.5,
        label='Langkah 1: IIR Notch Filter (50 Hz, Q=30)')
ax.set_title("Blok 3a: Hasil IIR Notch Filter 50 Hz (Eliminasi Noise Listrik PLN)", fontsize=11, fontweight='bold')
ax.set_xlabel("Waktu (detik)"); ax.set_ylabel("Amplitudo (a.u.)")
ax.legend(loc='upper right'); ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Standar Deviasi Sinyal Mentah   : {np.std(epoch_ecg_raw):.4f}")
print(f"Standar Deviasi Pasca-Notch     : {np.std(epoch_step1_notch):.4f}")

---
### 4.2 Blok 3b — Butterworth Bandpass Filter 0.5–40 Hz

**Tujuan**: Membatasi rentang spektrum frekuensi sinyal hanya pada pita informasi fisiologis ECG ($0.5–40\text{ Hz}$).

| Batas Frekuensi | Noise yang Dieliminasi | Alasan Fisiologis |
| :--- | :--- | :--- |
| **Highpass Cutoff ($0.5\text{ Hz}$)** | *Baseline wander*, pergerakan respirasi, keringat | Mengembalikan garis dasar ECG ke level nol datar |
| **Lowpass Cutoff ($40.0\text{ Hz}$)** | Kontraksi otot (*Electromyogram* / EMG), interferensi RF | Mempertahankan kompleks gelombang QRS ($10–25\text{ Hz}$) tanpa noise frekuensi tinggi |
| **Ordo Filter** | $N=2$ (efektif $N=4$ via `filtfilt`) | Respon frekuensi *maximally flat* (tanpa ripple di passband) |

In [ ]:
# ── Blok 3b: Butterworth Bandpass Filter 0.5 - 40 Hz ─────────────────────────

epoch_step2_bandpass = butterworth_bandpass_filter(
    epoch_step1_notch,
    fs=FS,
    lowcut=0.5,
    highcut=40.0,
    order=2
)

# Visualisasi Komparasi Bertahap
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Subplot 1: Overlay Pasca-Notch vs Pasca-Bandpass
ax1.plot(epoch_time, epoch_step1_notch, color='#2ca02c', alpha=0.5, lw=1.0, label='Pasca-Notch 50 Hz (Sebelum Bandpass)')
ax1.plot(epoch_time, epoch_step2_bandpass, color='#1f77b4', lw=1.5, label='Langkah 2: Butterworth Bandpass (0.5-40 Hz)')
ax1.set_title("Blok 3b: Hasil Butterworth Bandpass Filter (0.5 - 40 Hz, Ordo 2)", fontsize=11, fontweight='bold')
ax1.set_ylabel("Amplitudo (a.u.)"); ax1.legend(loc='upper right'); ax1.grid(True, linestyle='--', alpha=0.5)

# Subplot 2: Baseline Wander yang Berhasil Dihilangkan (Residu)
baseline_wander_removed = epoch_step1_notch - epoch_step2_bandpass
ax2.plot(epoch_time, baseline_wander_removed, color='#d62728', lw=1.0, label='Komponen Baseline Wander & High-Freq Noise yang Dihilangkan')
ax2.set_title("Komponen Noise / Baseline Drift yang Dieliminasi", fontsize=11, fontweight='bold')
ax2.set_xlabel("Waktu (detik)"); ax2.set_ylabel("Amplitudo Drift"); ax2.legend(loc='upper right'); ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---
### 4.3 Blok 3c — Min-Max Normalization [0, 1]

**Tujuan**: Menstandarkan amplitudo sinyal ECG ke interval seragam $[0, 1]$ sesuai metode yang digunakan pada **paper acuan utama arsitektur skripsi**:

> **1. Fan et al. (2021)** — *"An Adaptive Weight Learning-Based Multitask Deep Network for Continuous Blood Pressure Estimation Using Electrocardiogram Signals"* (Sensors, 2021, Hal. 5):
> *"To mitigate the different amplitudes of ECG signals, the min-max normalization technology is utilized for ECG signals to map the amplitudes ranging from 0 to 1."*

> **2. Li et al. (2020)** — *"Real-Time Cuffless Continuous Blood Pressure Estimation Using Deep Learning Model"* (Sensors, 2020, Hal. 5–6):
> *"We normalize the amplitude of ECG and PPG signals to the range [0, 1] to ease and robustify the process of comparisons and analysis... by dividing the subtraction of each sample and the minimum value with the subtraction of the maximum value and minimum value."*

**Formulasi Matematis (Fan et al., 2021 & Li et al., 2020)**:
$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min} + \epsilon}$$

di mana:
- $x_{\min} = \min(x)$ adalah amplitudo minimum pada segmen epoch 10 detik.
- $x_{\max} = \max(x)$ adalah amplitudo maksimum pada segmen epoch 10 detik.
- $\epsilon = 10^{-8}$ (*safety factor* pencegah pembagian nol jika sinyal datar).

In [ ]:
# ── Blok 3c: Min-Max Normalization [0, 1] ──────────────────────────────────────
# Sesuai paper acuan: Fan et al. (2021) & Li et al. (2020)

epoch_step3_final = minmax_scale(epoch_step2_bandpass)

# Visualisasi Sinyal Akhir Hasil Preprocessing
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# 1. Sinyal ECG Mentah Awal vs Sinyal Bersih Ternormalisasi
ax1.plot(epoch_time, epoch_ecg_raw, color='gray', alpha=0.5, lw=1.0, label='Kondisi Awal: ECG Mentah (Raw)')
ax1.set_title("Sebelum Preprocessing: Sinyal ECG Mentah", fontsize=11, fontweight='bold')
ax1.set_ylabel("Amplitudo Mentah"); ax1.legend(loc='upper right'); ax1.grid(True, linestyle='--', alpha=0.5)

# 2. Sinyal ECG Siap Training [0, 1]
ax2.plot(epoch_time, epoch_step3_final, color='#9467bd', lw=1.5, label='Langkah 3: Min-Max Scale [Fan et al., 2021] → Rentang [0, 1]')
ax2.axhline(0.0, color='black', linestyle=':', alpha=0.7, label='Batas Bawah (0.0)')
ax2.axhline(1.0, color='black', linestyle=':', alpha=0.7, label='Batas Atas (1.0)')
ax2.set_title("Blok 3c: Hasil Akhir Min-Max Normalization [0, 1] Sesuai Fan et al. (2021)", fontsize=11, fontweight='bold')
ax2.set_xlabel("Waktu (detik)"); ax2.set_ylabel("Amplitudo Ternormalisasi")
ax2.set_ylim(-0.05, 1.05)
ax2.legend(loc='upper right'); ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("Statistik Amplitudo Sinyal Akhir:")
print(f" - Nilai Minimum : {np.min(epoch_step3_final):.6f}  (Harus >= 0.0)")
print(f" - Nilai Maksimum : {np.max(epoch_step3_final):.6f}  (Harus <= 1.0)")
print(f" - Nilai Median   : {np.median(epoch_step3_final):.6f}")
print(f" - Nilai Mean     : {np.mean(epoch_step3_final):.6f}")

---
### 4.4 Verifikasi Pipeline End-to-End

Membuktikan bahwa eksekusi sekuensial tahap-demi-tahap (Blok 3a → 3b → 3c) menghasilkan luaran yang **100% identik** dengan fungsi `preprocess_ecg_pipeline()` gabungan.

In [ ]:
# Eksekusi via pipeline gabungan
epoch_from_pipeline = preprocess_ecg_pipeline(epoch_ecg_raw, fs=FS)

# Hitung perbedaan absolut maksimum
max_diff = np.max(np.abs(epoch_step3_final - epoch_from_pipeline))
is_identical = np.allclose(epoch_step3_final, epoch_from_pipeline, atol=1e-12)

print("Hasil Uji Verifikasi:")
print(f" - Max Absolute Difference : {max_diff:.2e}")
print(f" - Status Kesetaraan       : {'✅ 100% IDENTIK (VERIFIED)' if is_identical else '❌ TIDAK COCOK'}")

assert is_identical, "Peringatan: Output langkah bertahap tidak identik dengan pipeline gabungan!"

---
## 5. Blok 4 — Eksekusi Batch Preprocessing ke Seluruh Dataset & Simpan Part Preprocessed Baru

Setelah metode pemrosesan divalidasi secara visual pada 1 sampel di **Blok 3**, blok ini mengaplikasikan pipeline pra-pemrosesan ke **seluruh rekaman di setiap file part dataset** (`part_1.mat`, `part_2.mat`, dst.) dan menyimpannya sebagai berkas terkompresi baru `.npz`.

### Spesifikasi Berkas Hasil Pemrosesan (`.npz`):
- **Lokasi Penyimpanan**: `/kaggle/working/processed/{part_name}_preprocessed.npz`
- **Array `X`**: Bentuk `(N_epochs, 1250)` → Sinyal ECG ternormalisasi $[0, 1]$ siap latih.
- **Array `y`**: Bentuk `(N_epochs, 3)` → Target Ground Truth $[SBP, DBP, MAP]$ dalam $\text{mmHg}$.
- **Proses Filter Kualitas Sinyal**:
  1. Mengabaikan epoch sinyal datar (*flatline*, $\text{std} < 10^{-4}$).
  2. Memverifikasi kelayakan rentang fisiologis ($50 \le SBP \le 220\text{ mmHg}$ dan $30 \le DBP \le 140\text{ mmHg}$).
  3. Menerapkan *Zero-Phase IIR Notch 50 Hz*, *Butterworth Bandpass 0.5-40 Hz*, dan *Min-Max Normalization [0,1]*.

In [ ]:
# ── Blok 4: Eksekusi Batch Preprocessing Seluruh Part Dataset ──────────────────

# Konfigurasi Direktori Output
PROCESSED_DIR = '/kaggle/working/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Direktori Penyimpanan Dataset Hasil Preprocessing:")
print(f" -> {PROCESSED_DIR}\n")

processed_files_summary = []

for mat_path in mat_files:
    part_name = os.path.basename(mat_path)
    part_stem = os.path.splitext(part_name)[0]
    out_file_path = os.path.join(PROCESSED_DIR, f"{part_stem}_preprocessed.npz")
    
    print("=================================================================")
    print(f"Memproses Batch: {part_name} -> {os.path.basename(out_file_path)}")
    
    # Muat ulang dari disk (agar bisa gc setelah selesai)
    cell_array = load_mat_file(mat_path)
    if cell_array is None:
        continue
        
    num_records = get_num_records(cell_array)
    
    X_part = []
    y_part = []
    skipped = 0
    
    for r_idx in tqdm(range(num_records), desc=f"  Progress {part_stem}"):
        try:
            rec = extract_record_matrix(cell_array, r_idx)
        except Exception:
            skipped += 1
            continue
        if rec is None or rec.ndim != 2 or rec.shape[0] < 3:
            skipped += 1
            continue
            
        raw_ecg = rec[2, :]
        raw_abp = rec[1, :]
        n_samples = len(raw_ecg)
        
        # Segmentasi epoch 10 detik (1250 sampel), non-overlapping
        for start_idx in range(0, n_samples - EPOCH_LEN + 1, EPOCH_LEN):
            seg_ecg = raw_ecg[start_idx : start_idx + EPOCH_LEN]
            seg_abp = raw_abp[start_idx : start_idx + EPOCH_LEN]
            
            # Filter Kualitas 1: Abaikan sinyal datar / noise rusak
            if np.std(seg_ecg) < 1e-4:
                continue
            if np.isnan(seg_ecg).any() or np.isnan(seg_abp).any():
                continue
                
            # Filter Kualitas 2: Hitung Ground Truth BP beat-by-beat
            bp_info = compute_bp_metrics(seg_abp, fs=FS)
            if not bp_info['valid']:
                continue
                
            # Eksekusi Pipeline Pra-Pemrosesan ECG
            clean_ecg = preprocess_ecg_pipeline(seg_ecg, fs=FS)
            
            X_part.append(clean_ecg.astype(np.float32))
            y_part.append([bp_info['sbp'], bp_info['dbp'], bp_info['map']])
            
    X_part = np.array(X_part, dtype=np.float32)
    y_part = np.array(y_part, dtype=np.float32)
    
    if len(X_part) > 0:
        # Simpan ke format .npz terkompresi
        np.savez_compressed(
            out_file_path,
            X=X_part,
            y=y_part,
            fs=FS,
            part_name=part_name
        )
        size_mb = os.path.getsize(out_file_path) / (1024 * 1024)
        print(f"  [✅] Selesai: {len(X_part)} epoch valid -> {out_file_path} ({size_mb:.2f} MB)")
        processed_files_summary.append({
            'Part Sumber': part_name,
            'File Output': os.path.basename(out_file_path),
            'Jumlah Epoch Siap Latih': len(X_part),
            'Shape X (ECG)': str(X_part.shape),
            'Shape y (BP)': str(y_part.shape),
            'Ukuran File (MB)': round(size_mb, 2)
        })
    else:
        print(f"  [⚠️] Peringatan: Tidak ada epoch valid dari {part_name}")
    
    # Bebaskan memori
    del cell_array, X_part, y_part
    gc.collect()

df_proc_summary = pd.DataFrame(processed_files_summary)
print("\n--- Ringkasan Dataset Baru yang Telah Ter-Preprocessing ---")
display(df_proc_summary)

if len(df_proc_summary) > 0:
    total_epochs = df_proc_summary['Jumlah Epoch Siap Latih'].sum()
    total_size = df_proc_summary['Ukuran File (MB)'].sum()
    print(f"\n📊 TOTAL: {total_epochs:,} epoch siap latih | {total_size:.1f} MB")

---
### 5.1 Verifikasi Memuat Berkas Hasil Preprocessing (`.npz`) Siap Pakai

Memverifikasi integritas salah satu berkas `.npz` hasil olahan untuk memastikan tensor siap di-load oleh PyTorch DataLoader.

In [ ]:
# Uji Coba Memuat Salah Satu Part Dataset Hasil Preprocessing
npz_files = sorted(glob.glob(os.path.join(PROCESSED_DIR, '*.npz')))

if npz_files:
    sample_npz_path = npz_files[0]
    print(f"Memuat file hasil pra-pemrosesan: {sample_npz_path}")
    data_loaded = np.load(sample_npz_path)
    X_loaded = data_loaded['X']
    y_loaded = data_loaded['y']
    
    print(f" - Dimensi Tensor Input X (ECG) : {X_loaded.shape}  (N_epoch, 1250)")
    print(f" - Dimensi Target Output y (BP) : {y_loaded.shape}  (N_epoch, [SBP, DBP, MAP])")
    print(f" - Rentang Amplitudo Sinyal ECG : [{np.min(X_loaded):.4f}, {np.max(X_loaded):.4f}]")
    print(f" - Contoh Target BP Epoch ke-0  : SBP={y_loaded[0, 0]:.1f}, DBP={y_loaded[0, 1]:.1f}, MAP={y_loaded[0, 2]:.1f} mmHg")
    
    # Plot 1 Sampel dari Dataset Preprocessed Baru
    t = np.arange(EPOCH_LEN) / FS
    plt.figure(figsize=(14, 3.5))
    plt.plot(t, X_loaded[0], color='#1f77b4', lw=1.3, label='Sinyal ECG Bersih Ternormalisasi [0, 1]')
    plt.title(f"Sampel Epoch ke-0 dari Berkas Preprocessed ({os.path.basename(sample_npz_path)}) | Target: SBP={y_loaded[0,0]:.1f}, DBP={y_loaded[0,1]:.1f} mmHg", fontsize=11, fontweight='bold')
    plt.xlabel("Waktu (detik)"); plt.ylabel("Amplitudo [0, 1]")
    plt.grid(True, linestyle='--', alpha=0.5); plt.legend(loc='upper right')
    plt.tight_layout(); plt.show()
else:
    print("Jalankan cell pemrosesan batch di atas terlebih dahulu untuk mengekspor berkas .npz!")

---
## Ringkasan File Output

| File | Deskripsi |
|---|---|
| `processed/part_1_preprocessed.npz` s/d `part_12_preprocessed.npz` | Dataset ECG bersih [0,1] + ground truth BP siap latih |

File `.npz` dapat langsung dimuat di notebook training dengan:
```python
data = np.load('/kaggle/working/processed/part_1_preprocessed.npz')
X = data['X']  # shape (N, 1250)
y = data['y']  # shape (N, 3) = [SBP, DBP, MAP]
```

Atau unduh dari panel **Output** di sebelah kanan Kaggle notebook.